# Vì sao Random Forest, và vì sao báo cáo **hai** bộ số?

Notebook giải thích phần phân loại khuyết tật (giai đoạn 3) bằng số liệu:
vì sao chọn Random Forest thay vì SVM, và vì sao phải báo cáo cả **7 lớp** (có `background`)
lẫn **6 lớp** (bỏ `background`).

**Nguyên tắc**

- Huấn luyện/evaluate dùng đúng hàm trong `experiments/exp_classify.py` (một nguồn logic).
- Chia fold theo **`pair_id`** (GroupKFold): mọi blob của một ảnh chỉ nằm ở một phía,
  tránh rò rỉ vì các blob cùng ảnh rất giống nhau.
- Số chính thức đọc từ `experiments/classify_results.csv` + `classify_results_6cls.csv`.
- Không deep learning: cả SVM và RF đều dùng `cv2.ml`.
- Trước khi chụp ảnh: **Kernel → Restart & Run All**.


In [ ]:

from pathlib import Path
import csv
import os
import sys
import tempfile
import time

# Tìm project root (thư mục chứa dataset_utils.py) để notebook chạy được từ mọi nơi.
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "dataset_utils.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "experiments"))

import cv2
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

%matplotlib inline

CHART_DIR = Path(tempfile.gettempdir()) / "pcb_notebook_charts"
CHART_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams["figure.dpi"] = 110

import module_classification as mc
import exp_classify as ec

print("Project root :", PROJECT_ROOT)
print(f"{len(mc.FEATURE_NAMES)} đặc trưng:", ", ".join(mc.FEATURE_NAMES))
print("Các lớp      :", mc.CLASS_NAMES)


def to_display(image, max_width: int = 1000):
    """BGR/gray -> RGB và thu nhỏ để hiển thị."""
    if image.ndim == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    else:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    height, width = image.shape[:2]
    if width > max_width:
        scale = max_width / width
        image = cv2.resize(image, (max_width, int(height * scale)), interpolation=cv2.INTER_AREA)
    return image


def show_stack(items, figsize=(14, 4.2)):
    fig, axes = plt.subplots(len(items), 1, figsize=(figsize[0], figsize[1] * len(items)),
                             squeeze=False)
    for row, (image, title) in enumerate(items):
        axes[row, 0].imshow(to_display(image))
        axes[row, 0].set_title(title, fontsize=10)
        axes[row, 0].axis("off")
    plt.tight_layout()
    plt.show()


## 1. Bộ dữ liệu huấn luyện

Mỗi dòng = một blob phát hiện được trên ảnh test. `label` lấy từ `class_name` của cặp ảnh;
blob không khớp Ground Truth nào bị gán `background`.


In [ ]:
dataset_path = PROJECT_ROOT / "experiments" / "classifier_dataset.csv"
X, y, groups = ec.load_dataset(dataset_path)

counts = {name: int((y == name).sum()) for name in mc.CLASS_NAMES if (y == name).any()}
print(f"{len(y)} blob / {len(set(groups.tolist()))} cặp | {X.shape[1]} đặc trưng")
for name, count in counts.items():
    print(f"  {name:<16}{count:>6}  ({count / len(y):5.1%})")

fig, ax = plt.subplots(figsize=(10, 3.8))
names = list(counts)
ax.bar(names, [counts[n] for n in names],
       color=["#4C72B0"] * (len(names) - 1) + ["#C44E52"])
ax.set_ylabel("số blob")
ax.set_title("Phân bố lớp: 6 lớp lỗi gần cân bằng, background chỉ 1,8%")
ax.tick_params(axis="x", rotation=20)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
with dataset_path.open("r", encoding="utf-8-sig", newline="") as handle:
    dataset_rows = list(csv.DictReader(handle))

unmatched = sum(1 for row in dataset_rows if row["matched"] == "0")
print(f"Blob khớp Ground Truth : {len(dataset_rows) - unmatched}/{len(dataset_rows)} "
      f"({(len(dataset_rows) - unmatched) / len(dataset_rows):.1%})")
print(f"Blob không khớp (background): {unmatched} ({unmatched / len(dataset_rows):.1%})")
print("Ghép blob ↔ GT: tâm blob nằm trong hộp GT, hoặc IoU ≥ 0,1.")


## 2. Vì sao cần mô hình phi tuyến

14 đặc trưng chia 3 nhóm: hình học (`area`, `perimeter`, `aspect`, `extent`, `solidity`,
`circularity`), 7 mô-men Hu (bất biến với xoay/co giãn) và `mean_diff` (độ sáng của vết lỗi).
Biểu đồ dưới cho thấy các lớp **chồng lấn mạnh** — ranh giới không phải đường thẳng.


In [ ]:
classes = [name for name in mc.CLASS_NAMES if (y == name).any()]
palette = plt.cm.tab10(np.linspace(0, 1, len(classes)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
feature_pairs = (("solidity", "extent"), ("circularity", "perimeter"))
for ax, (fx, fy) in zip(axes, feature_pairs):
    for name, color in zip(classes, palette):
        mask = y == name
        ax.scatter(X[mask, mc.FEATURE_NAMES.index(fx)], X[mask, mc.FEATURE_NAMES.index(fy)],
                   s=8, alpha=0.45, color=color, label=name)
    ax.set_xlabel(fx)
    ax.set_ylabel(fy)
    ax.grid(alpha=0.25)
axes[1].set_yscale("log")
axes[0].legend(fontsize=8, markerscale=2)
fig.suptitle("Các lớp chồng lấn → cần mô hình phi tuyến (RF) thay vì ranh giới tuyến tính", fontsize=11)
plt.tight_layout()
plt.show()


## 3. So sánh mô hình (đọc CSV chính thức)

Cùng một cách chia fold, cùng đặc trưng; chỉ đổi họ mô hình.
SVM dùng RBF + z-score, C quét 1/10/100; RF 60 cây, độ sâu ≤ 12/20.


In [ ]:
def read_results(path):
    with Path(path).open("r", encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))


results_7 = read_results(PROJECT_ROOT / "experiments" / "classify_results.csv")
results_6 = read_results(PROJECT_ROOT / "experiments" / "classify_results_6cls.csv")

print(f"{'model':<14}{'F1 7 lớp':>10}{'acc 7':>9}{'F1 6 lớp':>10}{'acc 6':>9}")
print("-" * 52)
for row7, row6 in zip(results_7, results_6):
    print(f"{row7['config']:<14}{float(row7['macro_f1']):>10.4f}{float(row7['accuracy']):>9.4f}"
          f"{float(row6['macro_f1']):>10.4f}{float(row6['accuracy']):>9.4f}")


## 4. Chạy thật: GroupKFold 5 fold với RF 60 cây

Không đọc số có sẵn — đoạn này huấn luyện lại từ `classifier_dataset.csv` bằng đúng hàm
`run_fold` của `exp_classify.py` (standardize z-score tính trên train của từng fold).


In [ ]:
CONFIG = {"name": "RF_depth20", "kind": "rf", "depth": 20, "trees": 60}
FOLDS = 5
SEED = 0


def run_oof(X, y, groups, config, folds=FOLDS, seed=SEED):
    names = [n for n in mc.CLASS_NAMES if (y == n).any()]
    index_of = {n: i for i, n in enumerate(names)}
    y_idx = np.array([index_of[v] for v in y])
    fold_ids = ec.group_folds(groups, folds, seed)
    rng = np.random.default_rng(seed)
    oof = np.full(len(y), -1, dtype=int)
    for fold in range(folds):
        train_index = np.flatnonzero(fold_ids != fold)
        test_index = np.flatnonzero(fold_ids == fold)
        oof[test_index] = ec.run_fold(config, X[train_index], y_idx[train_index],
                                      X[test_index], rng, False)
    matrix = ec.confusion_matrix(y_idx, oof, len(names))
    rows = ec.per_class_scores(matrix, names)
    accuracy = float(np.trace(matrix) / matrix.sum()) if matrix.sum() else 0.0
    return names, matrix, rows, accuracy


start = time.perf_counter()
names_7, matrix_7, rows_7, accuracy_7 = run_oof(X, y, groups, CONFIG)
print(f"5 fold xong trong {time.perf_counter() - start:.0f}s")
print(f"7 lớp: macro F1 {ec.macro_f1(rows_7):.4f} | accuracy {accuracy_7:.4f}")


In [ ]:
header = f"{'lớp':<16}{'support':>9}{'precision':>11}{'recall':>9}{'F1':>8}"
print(header)
print("-" * len(header))
for row in rows_7:
    print(f"{row['class']:<16}{row['support']:>9}{row['precision']:>11.3f}"
          f"{row['recall']:>9.3f}{row['f1']:>8.3f}")
print(f"{'macro F1':<16}{'':>9}{'':>11}{'':>9}{ec.macro_f1(rows_7):>8.3f}")


In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 6))
image = ax.imshow(matrix_7, cmap="Blues")
ax.set_xticks(range(len(names_7)))
ax.set_yticks(range(len(names_7)))
ax.set_xticklabels([n[:12] for n in names_7], rotation=35, ha="right", fontsize=9)
ax.set_yticklabels(names_7, fontsize=9)
ax.set_xlabel("dự đoán")
ax.set_ylabel("thực tế")
for i in range(len(names_7)):
    for j in range(len(names_7)):
        value = matrix_7[i, j]
        ax.text(j, i, str(value), ha="center", va="center", fontsize=9,
                color="white" if value > matrix_7.max() * 0.5 else "black")
ax.set_title(f"RF 60 cây, sâu 20 — macro F1 {ec.macro_f1(rows_7):.3f}")
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


## 5. Bỏ `background`: bài toán 6 lớp

`background` chỉ chiếm 1,8% và **không học được** (xem mục 6). Bỏ nó ra khỏi bài toán
là cách báo cáo trung thực: con số 0,936 là của **6 lớp lỗi**, không phải của hệ thống
"loại được báo động giả".


In [ ]:
X6, y6, groups6 = ec.load_dataset(dataset_path, drop_background=True)
start = time.perf_counter()
names_6, matrix_6, rows_6, accuracy_6 = run_oof(X6, y6, groups6, CONFIG)
print(f"5 fold xong trong {time.perf_counter() - start:.0f}s")
print(f"6 lớp: macro F1 {ec.macro_f1(rows_6):.4f} | accuracy {accuracy_6:.4f}")
print()
for row in rows_6:
    print(f"  {row['class']:<16} F1 {row['f1']:.3f}  (precision {row['precision']:.3f}, "
          f"recall {row['recall']:.3f})")


## 6. Hạn chế phải nói rõ: `background` không học được


In [ ]:
background_col = names_7.index("background")
print("Số lần mô hình dự đoán 'background':", int(matrix_7[:, background_col].sum()))
print("Số blob 'background' thật          :", int(matrix_7[background_col, :].sum()))
print("→ Recall của background = 0: mô hình KHÔNG BAO GIỜ dự đoán lớp này.")
print("→ Nghĩa là bbox rác vẫn bị gán một nhãn lỗi; không được nói 'hệ thống loại được báo động giả'.")
print()
print("Nhầm lẫn chính (đọc từ ma trận):")
for i, name in enumerate(names_7):
    for j, other in enumerate(names_7):
        if i != j and matrix_7[i, j] >= 20:
            print(f"  {name} → {other}: {matrix_7[i, j]} blob")


## 7. Mô hình cuối và cách lưu

`model_classifier.pkl` chứa metadata + scaler; do OpenCV không pickle được model,
rừng cây được lưu riêng bằng `cv2.ml.RTrees_save` ra `model_classifier.rtrees.xml`
(`module_classification.save_model` / `load_model` xử lý cặp file này).


In [ ]:
import module_classification as mc

model = mc.load_model()
print("Cấu hình :", model["meta"]["config"])
print("Số dòng  :", model["meta"]["n_rows"], "| số cặp:", model["meta"]["n_pairs"])
print("Macro F1 :", round(model["meta"]["macro_f1"], 4), "| accuracy:", round(model["meta"]["accuracy"], 4))
print("Lớp      :", model["classes"])


---

**Ghi chú khi commit**

```powershell
jupyter nbconvert --clear-output --inplace experiments/notebooks/classification.ipynb
```

- Không ghi đè `classifier_dataset.csv`, `classify_results*.csv`, `model_classifier.*` từ notebook.
- Muốn sinh lại số chính thức: `build_classifier_dataset.py` → `exp_classify.py`.
